# Kulude nõude analüüs

See märkmik demonstreerib, kuidas luua agente, kes kasutavad pluginaid kohalike kviitungite piltidelt reisi kulude töötlemiseks, kulude nõude e-kirja genereerimiseks ja kulude andmete visualiseerimiseks sektordiagrammi abil. Agendid valivad funktsioonid dünaamiliselt ülesande konteksti põhjal.

Sammud:
1. OCR Agent töötleb kohalikku kviitungipilti ja eraldab reisi kulude andmed.
2. Email Agent genereerib kulude nõude e-kirja.

### Reisi kulude stsenaariumi näide:
Kujutage ette, et olete töötaja, kes reisib ärikohtumiseks teise linna. Teie ettevõttel on poliitika hüvitada kõik mõistlikud reisi seotud kulud. Siin on võimalikud reisi kulude jaotus:
- Transport:
Lennupilet edasi-tagasi teie kodulinnast sihtkoha linna.
Takso või sõidujagamise teenused lennujaama ja lennujaamast.
Kohalik transport sihtkoha linnas (nagu ühistransport, rendiautod või taksod).

- Majutus:
Kolmepäevane hotellimajutus keskmise hinnaklassi ärhotellis lähedal kohtumispaigale.

- Toidukulud:
Igapäevane toidutoetus hommiku-, lõuna- ja õhtusöögiks ettevõtte päevatasu poliitika alusel.

- Muud kulud:
Parkimistasud lennujaamas.
Internetikasutuse tasud hotellis.
Näpunäidised või väikesed teenustasud.

- Dokumentatsioon:
Esitate kõik kviitungid (lennud, taksod, hotell, toidud jne) ja täidetud kulude aruande hüvitamiseks.


## Nõutavate teekide importimine

Impordi märkmikuks vajalikud teegid ja moodulid.


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import base64
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

 ## Defineeri kulude mudelid

 Loo Pydantic mudel üksikute kulude jaoks ja ExpenseFormatter klass, mis teisendab kasutaja päringu struktureeritud kulude andmeteks.

 Iga kulu esitatakse vormingus:
 `{'date': '07-Mar-2025', 'description': 'lennureis sihtkohta', 'amount': 675.99, 'category': 'Transport'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Tööriistade määratlemine – e-kirja genereerimine

Loo tööriista funktsioon, mis genereerib e-kirja kuluhüvitise esitamiseks.
- See tööriist kasutab Microsoft Agent Frameworki `@tool` dekoratsiooni.
- See arvutab kulude kogusumma ja vormindab üksikasjad e-kirja sisuks.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Tööriist reisi kulude väljavõtmiseks tšekipiltidelt

Loo tööriista funktsioon reisi kulude väljavõtmiseks tšekipiltidelt.  
- See tööriist kasutab Microsoft Agent Frameworki `@tool` dekoratiivset märgendit.  
- See loeb tšekipildi, kodeerib selle base64 vormingusse ja tagastab andmete URI agendi analüüsimiseks.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Kulu töötlemine

Määratlege agendid ja ühendage need järjestikusse töövoogu, kasutades `WorkflowBuilder`.
- OCR-agent eraldab tšeki pildilt struktureeritud kulutuste andmed, kasutades tööriista `load_receipt_image`.
- Email-agent võtab välja eraldatud andmed ja genereerib professionaalse kuluhüvitise e-kirja, kasutades tööriista `generate_expense_email`.
- `WorkflowBuilder` koos `add_edge`-iga loob järjestikuse torujuhtme: OCR-agent → Email-agent.


In [ ]:
ocr_agent = await provider.create_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = await provider.create_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Peamine funktsioon

Koosta järjestikune töövoog ja käivita see tšeki pildi töötlemiseks ning kulunõude e-kirja genereerimiseks.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
